# GPU architecture tour

Peak-FLOPs and memory-bandwidth microbenchmarks, a compute-capability readout,
and a roofline scatter over matrix multiplies at varying arithmetic intensity.

**Learning objectives**
- Read compute capability, SM count, and advertised peak bandwidth from the device.
- Measure sustained memory bandwidth with a large contiguous copy.
- Measure sustained FP16 TFLOPs from a square matmul hot enough to amortise launch cost.
- Plot the roofline and classify a grid of matmul shapes as compute- or memory-bound.

**Papers**: Williams et al. *Roofline: An Insightful Visual Performance Model for Multicore Architectures*, CACM 2009.

**Hardware**: T4 or better. On CPU the GPU-specific checks are skipped but the arithmetic-intensity calculation still runs.
**Estimated runtime**: ≈10 min.


In [ ]:
from __future__ import annotations

import os
import random
import sys
import time
from pathlib import Path

REPO = Path.cwd()
while not (REPO / "scoring" / "harness.py").exists() and REPO != REPO.parent:
    REPO = REPO.parent
sys.path.insert(0, str(REPO))
sys.path.insert(0, str(REPO / "src"))

import numpy as np
import torch

from llm_infra_lab._utils import hardware_check, set_seed, timeit
from scoring.harness import Scorer

set_seed(0)
s = Scorer("07_gpu_01_gpu_architecture_tour")

info = hardware_check()
print(info)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
IS_CUDA = DEVICE.type == "cuda"


In [ ]:
# Advertised peak bandwidth and compute per device family (approximate,
# vendor datasheets). Used to compare microbenchmarks against theoretical.
DEVICE_PEAKS: dict[str, dict[str, float]] = {
    "T4":            {"peak_bw_gbps":  320.0, "peak_fp16_tflops":  65.0},
    "L4":            {"peak_bw_gbps":  300.0, "peak_fp16_tflops": 120.0},
    "A10G":          {"peak_bw_gbps":  600.0, "peak_fp16_tflops": 125.0},
    "A100":          {"peak_bw_gbps": 2039.0, "peak_fp16_tflops": 312.0},
    "H100":          {"peak_bw_gbps": 3350.0, "peak_fp16_tflops": 756.0},
    "V100":          {"peak_bw_gbps":  900.0, "peak_fp16_tflops": 125.0},
    "RTX 3090":      {"peak_bw_gbps":  936.0, "peak_fp16_tflops":  71.0},
    "RTX 4090":      {"peak_bw_gbps": 1008.0, "peak_fp16_tflops": 165.0},
}


def match_device(name: str) -> dict[str, float] | None:
    for key, peaks in DEVICE_PEAKS.items():
        if key.lower() in name.lower():
            return peaks
    return None


if IS_CUDA:
    dev_name = torch.cuda.get_device_name(0)
    peaks = match_device(dev_name)
    if peaks is None:
        print(f"No datasheet entry for {dev_name!r}; using A10G as a rough prior.")
        peaks = DEVICE_PEAKS["A10G"]
    print(f"Detected: {dev_name}")
    print(f"Datasheet peaks: {peaks}")
else:
    peaks = None


In [ ]:
def measure_bandwidth_gbps(n_bytes: int = 512 * 1024 * 1024, n_iter: int = 20) -> float:
    '''Average memory bandwidth from a contiguous float32 copy (in GB/s).'''
    if not torch.cuda.is_available():
        return 0.0
    n = n_bytes // 4
    a = torch.empty(n, device="cuda", dtype=torch.float32).uniform_()
    b = torch.empty_like(a)
    # Warmup.
    for _ in range(3):
        b.copy_(a)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        b.copy_(a)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    # Each copy reads n_bytes and writes n_bytes.
    return 2 * n_bytes * n_iter / dt / 1e9


bw_gbps = measure_bandwidth_gbps() if IS_CUDA else 0.0
print(f"measured bandwidth: {bw_gbps:.1f} GB/s")

if IS_CUDA and peaks is not None:
    s.assert_close(
        "bandwidth_within_30pct_of_peak",
        actual=bw_gbps,
        expected=peaks["peak_bw_gbps"],
        rtol=0.5,
    )
    s.benchmark(
        "bandwidth_at_least_40pct_peak",
        measured=bw_gbps,
        baseline=peaks["peak_bw_gbps"],
        must_beat=0.40,
        unit="GB/s",
    )
else:
    s.skip("bandwidth_within_30pct_of_peak", "no CUDA device available")
    s.skip("bandwidth_at_least_40pct_peak", "no CUDA device available")


In [ ]:
def measure_matmul_tflops(m: int = 4096, n_iter: int = 30, dtype: torch.dtype = torch.float16) -> float:
    '''Sustained TFLOPs from square matmul of side ``m`` in ``dtype``.'''
    if not torch.cuda.is_available():
        return 0.0
    a = torch.randn((m, m), device="cuda", dtype=dtype)
    b = torch.randn((m, m), device="cuda", dtype=dtype)
    for _ in range(3):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    t0 = time.perf_counter()
    for _ in range(n_iter):
        torch.matmul(a, b)
    torch.cuda.synchronize()
    dt = time.perf_counter() - t0
    flops = 2 * m**3
    return flops * n_iter / dt / 1e12


tflops = measure_matmul_tflops() if IS_CUDA else 0.0
print(f"measured FP16 matmul: {tflops:.1f} TFLOPs")

if IS_CUDA and peaks is not None:
    s.benchmark(
        "matmul_tflops_at_least_40pct_peak",
        measured=tflops,
        baseline=peaks["peak_fp16_tflops"],
        must_beat=0.40,
        unit="TFLOPs",
    )
else:
    s.skip("matmul_tflops_at_least_40pct_peak", "no CUDA device available")


In [ ]:
# Arithmetic intensity = FLOPs per byte. For square matmul M=N=K=m in fp16,
# FLOPs = 2m^3, bytes = 3 * m^2 * 2 = 6 m^2 (2 inputs + 1 output, fp16),
# so intensity = m / 3.
import matplotlib.pyplot as plt


def matmul_intensity(m: int, k: int, n: int, dtype_bytes: int = 2) -> float:
    flops = 2 * m * k * n
    bytes_ = dtype_bytes * (m * k + k * n + m * n)
    return flops / bytes_


# A grid of shapes: a tall-skinny vector-matrix (memory-bound at small bs),
# a square matmul (balanced at larger sizes), and a big square matmul (compute-bound).
SHAPES = [
    (1,      4096, 4096),
    (16,     4096, 4096),
    (128,    4096, 4096),
    (1024,   4096, 4096),
    (4096,   4096, 4096),
]
intensities = [matmul_intensity(m, k, n) for (m, k, n) in SHAPES]
print("arithmetic intensities:", {f"{m}x{k}x{n}": round(ai, 1) for (m, k, n), ai in zip(SHAPES, intensities)})

peak_bw = (peaks or {"peak_bw_gbps": bw_gbps or 320.0})["peak_bw_gbps"]
peak_compute = (peaks or {"peak_fp16_tflops": tflops or 65.0})["peak_fp16_tflops"]
ridge_intensity = peak_compute * 1e12 / (peak_bw * 1e9)
print(f"ridge arithmetic intensity: {ridge_intensity:.1f} FLOPs/byte")

fig, ax = plt.subplots(figsize=(7, 4))
x = np.logspace(-1, 3, 200)
y_mem  = peak_bw * 1e9 * x / 1e12
y_comp = np.full_like(x, peak_compute)
y = np.minimum(y_mem, y_comp)
ax.loglog(x, y, label=f"roofline (bw={peak_bw:.0f} GB/s, peak={peak_compute:.0f} TFLOPs)")
ax.axvline(ridge_intensity, color="tab:gray", linestyle=":", label=f"ridge @ {ridge_intensity:.0f}")
ax.scatter(intensities, [peak_compute * min(1.0, ai / ridge_intensity) for ai in intensities],
           color="tab:red", zorder=5, label="matmul shapes")
ax.set_xlabel("arithmetic intensity (FLOPs / byte)")
ax.set_ylabel("throughput (TFLOPs)")
ax.set_title("roofline: bandwidth-bound below ridge, compute-bound above")
ax.legend()
ax.grid(True, which="both", alpha=0.3)
plt.show()

# Classifications: m=1 must be memory-bound (intensity well below ridge);
# m=4096 must be compute-bound (intensity well above ridge).
mem_bound_shape = SHAPES[0]
comp_bound_shape = SHAPES[-1]
s.check(
    "small_batch_is_memory_bound",
    lambda: matmul_intensity(*mem_bound_shape) < ridge_intensity,
    msg=f"intensity={matmul_intensity(*mem_bound_shape):.1f} ridge={ridge_intensity:.1f}",
)
s.check(
    "large_square_is_compute_bound",
    lambda: matmul_intensity(*comp_bound_shape) > ridge_intensity,
    msg=f"intensity={matmul_intensity(*comp_bound_shape):.1f} ridge={ridge_intensity:.1f}",
)
s.check(
    "intensity_monotone_in_batch",
    lambda: all(intensities[i] <= intensities[i + 1] for i in range(len(intensities) - 1)),
    msg="arithmetic intensity should grow with batch for fixed k,n",
)


In [ ]:
# Basic device-capability sanity.
if IS_CUDA:
    cc = torch.cuda.get_device_capability(0)
    smp = torch.cuda.get_device_properties(0).multi_processor_count
    s.check(
        "compute_capability_detected",
        lambda: cc[0] >= 5,
        msg=f"cc={cc}",
    )
    s.check(
        "nontrivial_sm_count",
        lambda: smp >= 20,
        msg=f"{smp} SMs",
    )
else:
    s.skip("compute_capability_detected", "no CUDA device available")
    s.skip("nontrivial_sm_count", "no CUDA device available")


In [ ]:
s.summary()
s.save()
